<img src="../../shared/alchemi-banner-left.png" alt="NVIDIA ALCHEMI: AI for Chemistry and Materials Science" style="display:block;box-sizing:border-box;width:100%;max-width:100%;height:auto;">

# 02 · Data loading with Zarr

**Goal:** Write and reopen a three-record Zarr store, then apply the same interfaces to a 32-molecule NCI collection.

**Core concepts:** Zarr persistence, field ownership, custom `Reader` classes, `Dataset`, `InMemoryDataset`, `DataLoader`, `Batch`, indexing, validation, caching, and device placement.

**Prerequisites:** Complete the [Core playbook](../00-core-playbook/alchemi-core-playbook.ipynb) and be comfortable with `AtomicData` and `Batch` from [Part 01](../01-atomicdata-batch/atomicdata-and-batch.ipynb).

**Time to complete: about 30 minutes.**

> **Environment note:** Importing `nvalchemi.data` initializes Warp. On a CPU-only host, Warp may print CUDA driver-entry-point errors before these CPU examples run. The messages are left visible because they describe the host, not a successful CUDA setup. If you expected GPU execution, fix the NVIDIA driver/runtime boundary before continuing.

In [ ]:
from pathlib import Path

import helpers
import pandas as pd
import torch
from ase.io import read as read_extxyz
from IPython.display import display
from nvalchemi.data import (
    AtomicData,
    AtomicDataZarrReader,
    AtomicDataZarrWriter,
    Batch,
    DataLoader,
    Dataset,
    InMemoryDataset,
    Reader,
)
from pydantic import ValidationError

<details aria-label="New to ALCHEMI Toolkit?">
<summary>Where NVIDIA ALCHEMI fits (recap)</summary>

[ALCHEMI](https://developer.nvidia.com/cuda/cuda-x-libraries/alchemi) brings together Python building blocks, accelerated kernels, and deployable services for atomistic workflows.

- **ALCHEMI Toolkit** is a GPU-first Python framework with a unified, composable API for MLIPs and custom models. It provides GPU-native atomic data and batching, model adapters, MD classes, hooks, model training, and single- to multi-GPU pipelines.
[GitHub repo](https://github.com/NVIDIA/nvalchemi-toolkit) · [Docs](https://nvidia.github.io/nvalchemi-toolkit/) · Apache 2.0 license

- **Toolkit-Ops** supplies GPU-optimized, batched operations for neighbor lists, dynamics, dispersion, and electrostatics, with PyTorch and JAX bindings.
[GitHub repo](https://github.com/NVIDIA/nvalchemi-toolkit-ops) · [Docs](https://nvidia.github.io/nvalchemi-toolkit-ops/) · Apache 2.0 license

- **ALCHEMI NIM microservices** package supported atomistic workflows as cloud-ready services. The current catalog includes Batched Geometry Relaxation for structural optimization and Batched Molecular Dynamics. Self-hosting uses an NVIDIA AI Enterprise license.
 [Transparency card](https://docs.nvidia.com/nim/alchemi/alchemi-bgr/1.0.0/ai-transparency-card/overview.html) · [Docs](https://docs.nvidia.com/nim/alchemi/alchemi-bgr/latest/index.html)
</details>

Part 02 reuses Part 01's [molecule → one-graph `AtomicData` → packed-system `Batch` path](../01-atomicdata-batch/atomicdata-and-batch.ipynb#from-one-molecule-to-a-batch) and its field-ownership model.

**Course orientation:** The evolving shared curriculum map highlights Part 02; updates to the shared map appear here automatically.

<object data="../../shared/curriculum-map-02.svg" type="image/svg+xml" style="display:block;box-sizing:border-box;width:100%;max-width:100%;aspect-ratio:900/552;border:0;" aria-label="Interactive ALCHEMI Toolkit curriculum. Part 02 highlights data loading used by later model, simulation, training, and multi-GPU workflows."></object>

[Open the evolving Part 02 curriculum map directly](../../shared/curriculum-map-02.svg).

**Takeaway:** Part 02 turns one-graph `AtomicData` records into durable inputs that recover the same field ownership when a `DataLoader` packs them into `Batch` objects.

In [ ]:
molecule_table = helpers.load_molecule_manifest()
NCI_SOURCE = helpers.molecule_source_path()
runtime_owner, runtime_path = helpers.tutorial_workspace()
STORE = runtime_path / "three-records.zarr"
NCI_STORE = runtime_path / "nci-records.zarr"

## First: three small records

Start with hydrogen, water, and methane so every row is easy to inspect. Each record has atom-level `atomic_numbers` and `positions`, plus one system-level `record_id` that must survive writing, reading, and batching.

In [ ]:
synthetic_specs = [
    ([1, 1], [[-0.37, 0.00, 0.00], [0.37, 0.00, 0.00]]),
    ([8, 1, 1], [[0.00, 0.00, 0.00], [0.96, 0.00, 0.00], [-0.24, 0.93, 0.00]]),
    (
        [6, 1, 1, 1, 1],
        [[0.00, 0.00, 0.00], [0.63, 0.63, 0.63], [0.63, -0.63, -0.63],
         [-0.63, 0.63, -0.63], [-0.63, -0.63, 0.63]],
    ),
]
synthetic_records = [
    AtomicData(
        atomic_numbers=torch.tensor(numbers, dtype=torch.int32),
        positions=torch.tensor(positions, dtype=torch.float32),
    )
    for numbers, positions in synthetic_specs
]

In [ ]:
for record_id, record in enumerate(synthetic_records):
    record.add_system_property(
        "record_id", torch.tensor([record_id], dtype=torch.int64)
    )
source_record = synthetic_records[1]

In [ ]:
source_record_summary = {
    "boundary": "AtomicData before write",
    "type": type(source_record).__name__,
    "record IDs": [int(source_record.record_id.item())],
    "atomic_numbers": tuple(source_record.atomic_numbers.shape),
    "positions": tuple(source_record.positions.shape),
    "atom fields": sorted(source_record.node_properties),
    "system fields": sorted(source_record.system_properties),
    "ownership evidence": "AtomicData property views",
    "device": str(source_record.device),
}

In [ ]:
source_record_summary

`add_system_property(...)` marks `record_id` as one value per graph. That ownership is separate from tensor shape: a one-element tensor can still describe the whole system.

In [ ]:
source_batch = Batch.from_data_list(synthetic_records, device="cpu")
source_batch.num_graphs, source_batch.num_nodes, source_batch.batch_ptr.tolist()

`AtomicDataZarrWriter.write(...)` accepts one `AtomicData`, a list, or a `Batch`. The bulk `Batch` path writes concatenated atom fields and system fields in one operation while preserving graph boundaries.

In [ ]:
writer = AtomicDataZarrWriter(STORE)
writer.write(source_batch)

In [ ]:
del source_batch, source_record, synthetic_records, writer
source_objects_released = all(
    name not in globals()
    for name in ("synthetic_records", "source_record", "source_batch")
)
source_objects_released

In [ ]:
reader = AtomicDataZarrReader(STORE)
len(reader)

`source_objects_released` is `True`, yet the new reader exposes three logical records. This is a genuine reopen: the original `AtomicData` objects and source `Batch` are gone. `AtomicDataZarrReader` now owns random-access I/O and returns raw CPU tensor dictionaries plus metadata.

In [ ]:
raw_record, raw_metadata = reader.read(1)
raw_metadata

In [ ]:
field_table = pd.DataFrame(
    [
        {
            "field": name,
            "shape": tuple(raw_record[name].shape),
            "dtype": str(raw_record[name].dtype).removeprefix("torch."),
            "level": reader.field_levels[name],
            "unit": {"positions": "Å"}.get(name, "n/a"),
            "device": str(raw_record[name].device),
        }
        for name in ("atomic_numbers", "positions", "record_id")
    ]
)

In [ ]:
field_table.set_index("field")

In [ ]:
raw_record_summary = {
    "boundary": "Reader after reopen",
    "type": type(raw_record).__name__,
    "record IDs": [int(raw_record["record_id"].item())],
    "atomic_numbers": tuple(raw_record["atomic_numbers"].shape),
    "positions": tuple(raw_record["positions"].shape),
    "atom fields": sorted(
        name for name, level in reader.field_levels.items() if level == "atom"
    ),
    "system fields": sorted(
        name for name, level in reader.field_levels.items() if level == "system"
    ),
    "ownership evidence": "reader.field_levels",
    "device": str(raw_record["positions"].device),
    "metadata": raw_metadata,
}
raw_record_summary

`Dataset` adds `AtomicData` validation and target-device transfer. `DataLoader` requests record indices, uses batched reader calls, and yields graph-aware `Batch` objects.

In [ ]:
dataset = Dataset(reader, device="cpu", num_workers=2)
len(dataset)

In [ ]:
validated_record, validated_metadata = dataset[1]

In [ ]:
validated_record_summary = {
    "boundary": "Dataset indexing",
    "type": type(validated_record).__name__,
    "record IDs": [int(validated_record.record_id.item())],
    "atomic_numbers": tuple(validated_record.atomic_numbers.shape),
    "positions": tuple(validated_record.positions.shape),
    "atom fields": raw_record_summary["atom fields"],
    "system fields": raw_record_summary["system fields"],
    "ownership evidence": "reader.field_levels",
    "AtomicData system view": sorted(validated_record.system_properties),
    "position dtype": str(validated_record.positions.dtype),
    "device": str(validated_record.device),
    "metadata": validated_metadata,
}
validated_record_summary

`dataset[1]` recovers one validated `AtomicData` object: water with three-row atom fields, the stable `record_id`, and reader metadata. In this pinned release, direct `Dataset` indexing does not rebuild the custom `AtomicData.system_properties` view; `reader.field_levels` remains the durable ownership evidence. The `DataLoader` uses those levels to place `record_id` in the system-level keys of a packed `Batch`.

In [ ]:
loader = DataLoader(
    dataset,
    batch_size=3,
    shuffle=False,
    prefetch_factor=1,
    use_streams=False,
)

In [ ]:
first_batch = next(iter(loader))

In [ ]:
first_batch_summary = {
    "boundary": "DataLoader batching",
    "type": type(first_batch).__name__,
    "record IDs": first_batch.record_id.tolist(),
    "atomic_numbers": tuple(first_batch.atomic_numbers.shape),
    "positions": tuple(first_batch.positions.shape),
    "atom fields": sorted(first_batch.keys["node"]),
    "system fields": sorted(first_batch.keys["system"]),
    "ownership evidence": "Batch.keys",
    "graphs / atoms": (first_batch.num_graphs, first_batch.num_nodes),
    "device": str(first_batch.device),
    "metadata": "not emitted; record_id remains in the payload",
}
first_batch_summary

In [ ]:
comparison_fields = [
    "type",
    "record IDs",
    "positions",
    "system fields",
    "ownership evidence",
    "device",
]
payload_comparison = pd.DataFrame(
    [
        source_record_summary,
        raw_record_summary,
        validated_record_summary,
        first_batch_summary,
    ]
).set_index("boundary")[comparison_fields]

In [ ]:
payload_comparison

The `record_id`, position shape, field ownership, and CPU device remain inspectable across all four boundaries. For the validated item, ownership comes from `reader.field_levels` because direct `Dataset` indexing in this pinned release does not rebuild the custom `AtomicData.system_properties` view. `DataLoader` uses those levels when it packs three systems, so `Batch.keys["system"]` again contains `record_id`. Separate reader metadata does not travel with the batch.

<div style="display:block;box-sizing:border-box;width:100%;max-width:100%;min-width:0;background:#F2F3F1;color:#1B1E20;border:1px solid #D6D9D4;border-radius:8px;padding:0.78rem 0.95rem;line-height:1.45;overflow:hidden;overflow-wrap:anywhere;">
  <div style="color:#725B22;font-size:0.78rem;font-weight:700;letter-spacing:0.03em;margin-bottom:0.25rem;">💡 Highlight</div>
  <div style="min-width:0;font-size:0.98rem;overflow-wrap:anywhere;">Storage does not replace Toolkit's data model. A <code style="background:#E0E3DE;color:#111315;border:1px solid #CDD1CB;border-radius:4px;padding:0.05rem 0.28rem;font-weight:650;white-space:normal;overflow-wrap:anywhere;">Reader</code> supplies raw tensors, <code style="background:#E0E3DE;color:#111315;border:1px solid #CDD1CB;border-radius:4px;padding:0.05rem 0.28rem;font-weight:650;white-space:normal;overflow-wrap:anywhere;">Dataset</code> restores validated graph records, and <code style="background:#E0E3DE;color:#111315;border:1px solid #CDD1CB;border-radius:4px;padding:0.05rem 0.28rem;font-weight:650;white-space:normal;overflow-wrap:anywhere;">DataLoader</code> emits the same <code style="background:#E0E3DE;color:#111315;border:1px solid #CDD1CB;border-radius:4px;padding:0.05rem 0.28rem;font-weight:650;white-space:normal;overflow-wrap:anywhere;">Batch</code> used in Part 01.</div>
</div>

Which object owns each boundary?

<div style="display:block;box-sizing:border-box;width:100%;max-width:100%;min-width:0;background:#151A1F;color:#F3F4F6;border:1px solid #30363D;border-radius:8px;padding:0.82rem 0.95rem;line-height:1.45;overflow:hidden;overflow-wrap:anywhere;">
  <div style="color:#76B900;font-size:0.76rem;font-weight:700;letter-spacing:0.06em;margin-bottom:0.3rem;">ALCHEMI TOOLKIT API</div>
  <code style="display:block;color:#FFFFFF;font-size:1rem;font-weight:650;white-space:normal;overflow-wrap:anywhere;">reader.read(index) -&gt; (dict[str, Tensor], metadata)</code>
  <code style="display:block;color:#FFFFFF;font-size:1rem;font-weight:650;white-space:normal;overflow-wrap:anywhere;">dataset[index] -&gt; (AtomicData, metadata)</code>
  <code style="display:block;color:#FFFFFF;font-size:1rem;font-weight:650;white-space:normal;overflow-wrap:anywhere;">DataLoader(dataset, batch_size=...) -&gt; Iterator[Batch]</code>
  <div style="display:grid;grid-template-columns:max-content minmax(0,1fr);column-gap:0.65rem;row-gap:0.18rem;min-width:0;color:#CDD2D8;margin-top:0.5rem;font-size:0.92rem;">
    <strong style="color:#F3F4F6;">Accepts</strong><span style="min-width:0;overflow-wrap:anywhere;">a logical record index, then a batch-loadable dataset.</span>
    <strong style="color:#F3F4F6;">Returns</strong><span style="min-width:0;overflow-wrap:anywhere;">raw CPU tensors, validated graph data, then graph-aware batches.</span>
  </div>
</div>

## Where does a stored record become graph data?

```mermaid
%%{init: {"theme":"base","flowchart":{"curve":"linear","nodeSpacing":18,"rankSpacing":28},"themeVariables":{"fontFamily":"NVIDIA Sans, Arial, sans-serif","background":"#000000","lineColor":"#7C8794"}}}%%
flowchart LR
  accTitle: Stored records become graph batches
  accDescr: Zarr arrays pass through Reader and Dataset before DataLoader collates graph-aware Batch objects.
  Z("Zarr arrays<br/>fields + pointers") -->|"read / read_many"| R("Reader<br/>raw CPU tensors")
  R -->|"model_validate"| D("Dataset<br/>AtomicData")
  D -->|"collate requested indices"| L("DataLoader<br/>Batch")
  classDef storage fill:#3A332C,color:#F3F4F6,stroke:#4A423A,stroke-width:1px;
  classDef current fill:#76B900,color:#050505,stroke:#76B900,stroke-width:1px;
  class Z storage;
  class R,D,L current;
  linkStyle default stroke:#7C8794,stroke-width:1.25px;
```

The reader owns storage I/O; validation starts at `Dataset`, and `DataLoader` preserves the requested graph grouping in each `Batch`.

In [ ]:
dataset.close()
del raw_record, validated_record, first_batch

## Then: stream the NCI collection

The curated [NCI Atlas](https://github.com/Honza-R/NCIAtlas) subset ([CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)) contains 32 neutral H/C/N/O molecules and 322 atoms. We will read extxyz frames on demand, validate them, write four Zarr batches, and reopen the result.

The manifest remains separate from tensor storage: labels and formulas help us inspect records, while `record_id` is the system-level tensor that travels through the Toolkit data path.

In [ ]:
{
    "molecules": len(molecule_table),
    "atoms": int(molecule_table["atoms"].sum()),
    "elements": "H, C, N, O",
    "formal charges": sorted(molecule_table["charge"].unique().tolist()),
}

In [ ]:
molecule_table.loc[[0, 23, 31], ["label", "formula", "atoms", "source"]]

In [ ]:
class ExtXYZReader(Reader):
    """Read one extxyz frame as a raw CPU tensor dictionary."""

    def __init__(self, path: Path, manifest: pd.DataFrame):
        super().__init__()
        self.path, self.table = path, manifest.reset_index(drop=True)

    def __len__(self):
        return len(self.table)

    @property
    def field_levels(self):
        return {
            "atomic_numbers": "atom", "positions": "atom", "record_id": "system"
        }

    def _load_sample(self, index):
        atoms = read_extxyz(self.path, index=index)
        return {
            "atomic_numbers": torch.tensor(atoms.numbers, dtype=torch.int32),
            "positions": torch.tensor(atoms.positions, dtype=torch.float32),
            "record_id": torch.tensor([index], dtype=torch.int64),
        }

    def _get_sample_metadata(self, index):
        return self.table.loc[index, ["label", "formula", "source"]].to_dict()

In [ ]:
extxyz_reader = ExtXYZReader(NCI_SOURCE, molecule_table)
raw_record, raw_metadata = extxyz_reader.read(23)
{
    "metadata": raw_metadata,
    "positions": tuple(raw_record["positions"].shape),
    "record_id": int(raw_record["record_id"].item()),
    "device": str(raw_record["positions"].device),
}

### Reader stops at raw CPU tensors

The adapter implements four `Reader` extension points: length, field levels, one-record loading, and metadata. `Reader.read(...)` adds the index and returns CPU tensors; it does not validate `AtomicData` or choose a target device. `Dataset` reaches this adapter through the public `read_many(...)` path, whose default implementation calls `_load_sample(...)` in requested order.

| Object | Responsibility in this lesson |
|---|---|
| `Reader` | Source I/O, raw CPU tensors, metadata, requested order |
| `Dataset` | `AtomicData` validation, target device, threaded prefetch |
| `DataLoader` | Index groups, collation, graph-aware `Batch` output |

In [ ]:
nci_dataset = Dataset(extxyz_reader, device="cpu", num_workers=2)
len(nci_dataset), nci_dataset.target_device

In [ ]:
validated_record, validated_metadata = nci_dataset[23]
{
    "type": type(validated_record).__name__,
    "label": validated_metadata["label"],
    "formula": validated_metadata["formula"],
    "positions": tuple(validated_record.positions.shape),
    "record_id": int(validated_record.record_id.item()),
    "device": str(validated_record.device),
}

## Validate where raw tensors enter Toolkit

The extxyz reader produced a consistent 13-atom phenol record. To locate the validation boundary, make one deliberately broken reader that removes a single position row while leaving `atomic_numbers` unchanged.

In [ ]:
class MissingPositionReader(Reader):
    """Return one record with a shortened positions tensor."""

    def __init__(self, sample, field_levels):
        super().__init__()
        self.sample = sample
        self._field_levels = field_levels

    def __len__(self):
        return 1

    @property
    def field_levels(self):
        return self._field_levels

    def _load_sample(self, index):
        sample = dict(self.sample)
        sample["positions"] = sample["positions"][:-1]
        return sample

The reader can return this malformed dictionary because its job is I/O. Indexing through `Dataset` is the point where the inconsistent graph is rejected.

In [ ]:
invalid_reader = MissingPositionReader(raw_record, extxyz_reader.field_levels)
invalid_dataset = Dataset(invalid_reader, device="cpu")
try:
    invalid_dataset[0]
except ValidationError as error:
    validation_message = error.errors()[0]["msg"]
finally:
    invalid_dataset.close()
validation_message

The raw record changed from `torch.Size([13, 3])` to `torch.Size([12, 3])`. `Dataset` calls `AtomicData.model_validate(...)` and reports “expected 13, got 12.” This example demonstrates one invariant only: the atom-level `positions` row count agrees with the graph's atom count.

Now write the valid NCI collection incrementally. Each loader result is already a `Batch`, so the first one starts the store and later ones append.

In [ ]:
nci_source_loader = DataLoader(
    nci_dataset,
    batch_size=8,
    shuffle=False,
    prefetch_factor=2,
    use_streams=False,
)
nci_writer = AtomicDataZarrWriter(NCI_STORE)
for batch_number, batch in enumerate(nci_source_loader):
    if batch_number == 0:
        nci_writer.write(batch)
    else:
        nci_writer.append(batch)

In [ ]:
nci_dataset.close()
del nci_source_loader, nci_writer, raw_record, validated_record
nci_reader = AtomicDataZarrReader(NCI_STORE)
len(nci_reader)

In [ ]:
EXAMPLE_IDS = [31, 0, 23]
selected_records = nci_reader.read_many(EXAMPLE_IDS)

`read_many(...)` preserves the requested order even when indices are non-sorted. The reader metadata reports each logical index, while the stored `record_id` provides the same identity inside the tensor payload.

In [ ]:
selected_rows = [
    {
        "requested index": requested_id,
        "reader index": metadata["index"],
        "stored record_id": int(payload["record_id"].item()),
        "molecule": molecule_table.loc[requested_id, "label"],
        "atoms": int(payload["atomic_numbers"].numel()),
    }
    for requested_id, (payload, metadata) in zip(
        EXAMPLE_IDS,
        selected_records,
        strict=True,
    )
]
pd.DataFrame(selected_rows)

In [ ]:
record_indices = range(len(nci_reader))
atom_counts = [nci_reader.get_metadata(index)[0] for index in record_indices]
atoms_ptr = [0, *torch.tensor(atom_counts).cumsum(0).tolist()]
figure = helpers.plot_record_layout(atom_counts, atoms_ptr)
plot_alt = "NCI record sizes and cumulative Zarr atom-row boundaries."
display(helpers.figure_to_html(figure, plot_alt))

**Figure description:** NCI record sizes and cumulative Zarr atom-row boundaries. The left panel shows why graph batches are ragged: molecules contain different numbers of atoms. The right panel is the corresponding cumulative `atoms_ptr`; consecutive boundaries delimit each record's rows in the concatenated atom arrays.

## Stream the reopened store

The on-demand dataset requests records from Zarr. Four loader outputs should cover all 32 graphs and all 322 atoms, with `record_id` still classified at system level.

In [ ]:
nci_stored_dataset = Dataset(nci_reader, device="cpu", num_workers=2)
stored_loader = DataLoader(
    nci_stored_dataset,
    batch_size=8,
    shuffle=False,
    prefetch_factor=2,
    use_streams=False,
)

In [ ]:
batch_stream = iter(stored_loader)

Summarize each emitted `Batch` by graph count, atom count, record range, type, and device. The two assertions check the complete collection rather than one representative batch.

In [ ]:
stream_rows = []
first_streamed = None
for number, batch in enumerate(batch_stream, start=1):
    if first_streamed is None:
        first_streamed = batch
    stream_rows.append(
        {
            "batch": number,
            "graphs": batch.num_graphs,
            "atoms": batch.num_nodes,
            "first ID": int(batch.record_id[0].item()),
            "last ID": int(batch.record_id[-1].item()),
            "type": type(batch).__name__,
            "device": str(batch.device),
        }
    )
stream_table = pd.DataFrame(stream_rows)
assert first_streamed is not None
assert int(stream_table["graphs"].sum()) == 32
assert int(stream_table["atoms"].sum()) == 322

In [ ]:
stream_table.set_index("batch")

In [ ]:
stream_summary = {
    "batch types": sorted(stream_table["type"].unique().tolist()),
    "graphs": int(stream_table["graphs"].sum()),
    "atoms": int(stream_table["atoms"].sum()),
    "system fields": sorted(first_streamed.keys["system"]),
}
stream_summary

## Choose on-demand or resident loading

`Dataset` reads requested records from the backing store and moves validated samples to its target device. Its `num_workers` creates threads for prefetch, not forked worker processes.

With `InMemoryDataset(reader=...)`, the reader-backed cache is built on CPU; `device` controls the device of emitted samples and batches. For a GPU-resident cache, pass a prebuilt GPU `in_memory_batch` instead. Resident loading suits repeated passes over a collection that fits in the chosen memory.

| Choice | Storage owner | Cache owner | Emitted device |
|---|---|---|---|
| `Dataset` | `Reader` | Prefetched records | Resolved `target_device` |
| `InMemoryDataset(reader=...)` | Reader during construction | CPU `in_memory_batch` | Resolved `target_device` |
| `InMemoryDataset(in_memory_batch=...)` | Caller | Caller-provided batch | Resolved `target_device` |

In [ ]:
resident_reader = AtomicDataZarrReader(NCI_STORE)
resident_dataset = InMemoryDataset(
    reader=resident_reader,
    chunk_size=16,
    device="cpu",
)
{
    "graphs": resident_dataset.in_memory_batch.num_graphs,
    "atoms": resident_dataset.in_memory_batch.num_nodes,
    "cache device": str(resident_dataset.in_memory_batch.device),
    "emitted target": str(resident_dataset.target_device),
}

In [ ]:
resident_loader = DataLoader(
    resident_dataset,
    batch_size=8,
    shuffle=False,
    prefetch_factor=2,
    use_streams=False,
)

In [ ]:
resident_first = next(iter(resident_loader))

In [ ]:
resident_parity = {
    "same IDs": torch.equal(resident_first.record_id, first_streamed.record_id),
    "same boundaries": torch.equal(
        resident_first.batch_ptr,
        first_streamed.batch_ptr,
    ),
    "same atomic numbers": torch.equal(
        resident_first.atomic_numbers,
        first_streamed.atomic_numbers,
    ),
    "same positions": torch.equal(
        resident_first.positions,
        first_streamed.positions,
    ),
    "cache device": str(resident_dataset.in_memory_batch.device),
    "emitted device": str(resident_first.device),
}
resident_parity

## Try it: request three records

Change `requested_ids` to any three distinct integers from 0 through 31, keeping them out of sorted order. Check both forms of identity: reader metadata should preserve requested indices, and the system-level `record_id` values should return in the same order. The labels come from the manifest, not from Zarr tensor metadata.

In [ ]:
requested_ids = [23, 31, 0]
requested_records = nci_reader.read_many(requested_ids)
loaded_ids = [
    int(record["record_id"].item()) for record, _ in requested_records
]
loaded_indices = [metadata["index"] for _, metadata in requested_records]
requested_labels = molecule_table.loc[requested_ids, "label"].tolist()
loaded_labels = molecule_table.loc[loaded_ids, "label"].tolist()
assert loaded_ids == requested_ids
assert loaded_indices == requested_ids
assert loaded_labels == requested_labels
f"Success: requested order is preserved: {loaded_labels}"

In [ ]:
nci_stored_dataset.close()
resident_dataset.close()
runtime_owner.cleanup()

## Where persisted records come from

This lesson wrote prepared `Batch` objects directly with `AtomicDataZarrWriter`. Dynamics workflows can use `nvalchemi.dynamics.ZarrData` to write snapshots as a simulation runs. It delegates serialization to the same writer and appends batches to the store.

Continue with NVIDIA's official [Dataset and DataLoader guide](https://nvidia.github.io/nvalchemi-toolkit/userguide/datapipes.html) or the complete [trajectory Zarr I/O example](https://nvidia.github.io/nvalchemi-toolkit/examples/intermediate/02_trajectory_zarr_io.html).

## Recap

### What you learned

- `AtomicDataZarrWriter.write(...)` starts a store; `append(...)` extends it without changing field ownership or graph boundaries.
- `Reader.read(...)` and `read_many(...)` return ordered raw CPU tensors plus metadata.
- A custom `Reader` only adapts source I/O. `Dataset` validates records and chooses their emitted device.
- `DataLoader` collates requested records into graph-aware `Batch` objects.
- `InMemoryDataset` uses a resident cache to avoid repeated source reads while keeping the loader interface unchanged.
- One system-level `record_id` survives synthetic and NCI paths, including non-sorted indexing.

### How we will use this

Part 03 keeps the recovered `Batch` interface fixed while selecting and composing model adapters. Later dynamics lessons can write snapshots through `ZarrData` and replay them through the same path.

Next: [Model interfaces and composition](../03-model-interfaces-composition/model-interfaces-composition.ipynb).